In [ ]:
from google.colab import files
import pandas as pd
import datetime

print("Upload the new data file:")
new_file = files.upload()

print("\nUpload the jan file (connected to Power BI):")
jan_file = files.upload()

In [ ]:
new_name = list(new_file.keys())[0]
jan_name = list(jan_file.keys())[0]

df = pd.read_excel(new_name)
jan = pd.read_excel(jan_name)

print(f"New file: {len(df)} rows")
print(f"Jan file: {len(jan)} rows")

In [ ]:
# Check: does the file have repeated columns (1-5) or not?
if 'Healthcare Worker Type (1)' in df.columns:
    print("File has repeated columns (1-5) — combining them...")

    base = ['DEPARTMENT', 'AUDIT DATE', 'ADUIT TIME']
    frames = []
    for i in range(1, 6):
        temp = df[base + [
            f'Healthcare Worker Type ({i})',
            f'Opp. Indication ({i})',
            f'HH Action ({i})'
        ]].copy()
        temp.columns = base + ['Healthcare Worker Type', 'Opportunity', 'Hand Hygiene Action']
        frames.append(temp)

    result = pd.concat(frames, ignore_index=True)
    result.dropna(subset=['Healthcare Worker Type', 'Opportunity', 'Hand Hygiene Action'], how='all', inplace=True)

else:
    print("File is already combined — skipping to cleaning...")
    result = df.copy()

# === Cleaning (runs in both cases) ===

# Fill empty cells using forward fill
result['DEPARTMENT'] = result['DEPARTMENT'].ffill()
result['ADUIT TIME'] = result['ADUIT TIME'].ffill()
result['AUDIT DATE'] = result['AUDIT DATE'].ffill()

# Strip whitespace from text columns
result['Opportunity'] = result['Opportunity'].str.strip()

# Fix date formats
def fix_date(val):
    if pd.isna(val):
        return val
    if isinstance(val, datetime.datetime):
        return val
    s = str(val).strip()
    if len(s) == 8 and s.isdigit():
        return pd.to_datetime(s, format='%m%d%Y')
    return pd.NaT

result['AUDIT DATE'] = result['AUDIT DATE'].apply(fix_date)
result['AUDIT DATE'] = pd.to_datetime(result['AUDIT DATE'], errors='coerce')
result['AUDIT DATE'] = result['AUDIT DATE'].dt.strftime('%m/%d/%Y')

# Convert Opportunity abbreviations to full text
opp_map = {
    'bef-pat': 'Before contact with the patient',
    'aft-pat': 'After contact with the patient',
    'aft.p.surr': "After contact with the patient's surroundings",
    'bef-asept': 'Before aseptic/clean procedure',
    'aft-b.f.': 'After blood/body fluids exposure risk'
}
result['Opportunity'] = result['Opportunity'].map(opp_map).fillna(result['Opportunity'])

# Format jan dates
jan['AUDIT DATE'] = pd.to_datetime(jan['AUDIT DATE']).dt.strftime('%m/%d/%Y')

# Append new data to jan file
combined = pd.concat([jan, result], ignore_index=True)

print(f"\nOriginal jan rows: {len(jan)}")
print(f"New rows added: {len(result)}")
print(f"Total rows: {len(combined)}")
print(f"Columns: {list(combined.columns)}")
combined.head(10)

In [ ]:
combined.to_excel('jan.xlsx', index=False, sheet_name='JAN 26')
files.download('jan.xlsx')
print("Done! File downloaded to your device")